# IBA — NLP with Deep Learning | Assignment 04
## Track 2, Option D: Ablation Study — LoRA Fine-Tuning on Reasoning Datasets

**Course:** NLP with Deep Learning — IBA Karachi  
**Task:** Fine-tune `Qwen/Qwen3-0.6B` on two reasoning datasets using LoRA (via Unsloth/TRL), then compare the resulting adapters using `Kimi-K2-Instruct` as an LLM judge across a 30-prompt benchmark.

This self-contained notebook executes the complete experimental pipeline across four sequential phases:

| Phase | Description | Estimated Time (T4 GPU) |
|-------|-------------|------------------------|
| 1 | Baseline evaluation — `Qwen3-0.6B` and `Qwen3-1.7B` on 30 benchmark prompts | ~30 min |
| 2 | 5-trial LoRA ablation — Dataset A: Crownelius (Claude Opus 4.6 reasoning) | ~3–4 hr |
| 3 | 5-trial LoRA ablation — Dataset B: Sky-T1 (QwQ-32B-Preview reasoning) | ~3–4 hr |
| 4 | 3-way comparative evaluation and summary report tables | ~30 min |

**Estimated total runtime: ~7–9 hours. Fits within Kaggle's 12-hour GPU limit.**

---

**Setup instructions before running:**
1. Enable GPU: *Settings → Accelerator → GPU T4 x1*
2. Enable Internet access: *Settings → Internet → On*
3. Add HuggingFace token: *Add-ons → Secrets → key `HF_TOKEN`*

All intermediate outputs are saved to `/kaggle/working/` after each phase to preserve progress.

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Install unsloth with CUDA 12.4 + torch 2.6.0 wheels (also installs compatible transformers)
pip('unsloth[cu124-torch260] @ git+https://github.com/unslothai/unsloth.git')
# Remaining deps — transformers intentionally omitted (unsloth pins a compatible version)
pip('trl', 'peft', 'datasets', 'accelerate', 'bitsandbytes', 'huggingface_hub', 'openai')
# Force-pin torchao LAST so no transitive dep can upgrade it to a version that calls
# torch.utils._pytree.register_constant (added in torch 2.6; absent on Kaggle T4 image)
pip('torchao==0.6.1', '--force-reinstall', '--no-deps')

In [ ]:
import os, json, time, re, gc, torch, pandas as pd
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from openai import OpenAI

# ── HuggingFace token from Kaggle Secrets ─────────────────────────────────────
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')

# ── Global config ─────────────────────────────────────────────────────────────
BASE_MODEL   = 'Qwen/Qwen3-0.6B'
DATASET_A_ID = 'Crownelius/Opus-4.6-Reasoning-3300x'
DATASET_B_ID = 'NovaSky-AI/Sky-T1_data_17k'
JUDGE_MODEL  = 'moonshotai/Kimi-K2-Instruct:novita'
JUDGE_CLIENT = OpenAI(base_url='https://router.huggingface.co/v1', api_key=HF_TOKEN)

MAX_SEQ_LEN   = 1536        # reduced from 2048 for T4 16 GB VRAM
SUBSET_B_SIZE = 2500        # Sky-T1 rows to use
WORK_DIR      = Path('/kaggle/working')
ADAPTERS_A    = WORK_DIR / 'adapters_A'
ADAPTERS_B    = WORK_DIR / 'adapters_B'
ADAPTERS_A.mkdir(exist_ok=True)
ADAPTERS_B.mkdir(exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_fp16 = torch.cuda.is_available() and not torch.cuda.is_bf16_supported()
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'fp16={use_fp16} | bf16={use_bf16}')

In [ ]:
# ── Shared helper functions ────────────────────────────────────────────────────

JUDGE_PROMPT_TEMPLATE = """\
You are an expert evaluator of reasoning quality.

QUESTION:
{question}

REFERENCE ANSWER (Gold):
{gold}

CANDIDATE RESPONSE:
{response}

Evaluate the candidate response on REASONING QUALITY:
5 = Correct answer, clear step-by-step reasoning, no errors
4 = Correct answer, reasoning mostly clear with minor gaps
3 = Partially correct or reasoning has notable gaps
2 = Wrong answer but shows some reasoning attempt
1 = Wrong answer, no meaningful reasoning

Respond in this exact format (nothing else):
SCORE: <number 1-5>
REASON: <one sentence justification>"""


def _parse_judge(out):
    m = re.search(r'SCORE\s*:\s*([1-5])', out, re.IGNORECASE)
    r = re.search(r'REASON\s*:\s*(.+)', out, re.IGNORECASE | re.DOTALL)
    if not m:
        raise ValueError(f'No SCORE in judge output: {out!r}')
    return int(m.group(1)), (r.group(1).strip() if r else out.strip())


def judge_response(question, gold, response):
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        question=question, gold=gold, response=response)
    for attempt in range(3):
        try:
            out = JUDGE_CLIENT.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=120, temperature=0.0,
            ).choices[0].message.content.strip()
            return _parse_judge(out)
        except Exception as e:
            time.sleep(2 * (attempt + 1))
    return 0, 'ERROR after retries'


def _apply_chat_template(tokenizer, messages):
    # apply_chat_template with return_tensors='pt' always returns a Tensor
    return tokenizer.apply_chat_template(
        messages, return_tensors='pt', add_generation_prompt=True)


def load_sft_model(adapter_path):
    model, tokenizer = FastLanguageModel.from_pretrained(
        str(adapter_path), max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True, token=HF_TOKEN)
    FastLanguageModel.for_inference(model)
    return model, tokenizer


def generate_response(model, tokenizer, prompt_text,
                      system='You are a helpful reasoning assistant. Think step by step.'):
    messages = [{'role': 'system', 'content': system},
                {'role': 'user',   'content': prompt_text}]
    input_ids = _apply_chat_template(tokenizer, messages).to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids, max_new_tokens=512, temperature=0.1,
            do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(
        out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()


def eval_model_on_prompts(model, tokenizer, PROMPTS, gold_answers, label):
    records = []
    for p in PROMPTS:
        resp  = generate_response(model, tokenizer, p['prompt'])
        score, reason = judge_response(p['prompt'], gold_answers[p['id']], resp)
        records.append({'label': label, 'prompt_id': p['id'], 'type': p['type'],
                        'score': score, 'reason': reason, 'response': resp})
        print(f'  [{label}] {p["id"]}: {score}/5')
        time.sleep(0.5)
    return records


TRIAL_CONFIGS = [
    {'name': 'T1', 'data_pct': 0.25, 'rank': 16,
     'target_modules': ['q_proj','k_proj','v_proj','o_proj'],
     'lr': 2e-4, 'batch': 2, 'grad_accum': 4, 'epochs': 1,
     'purpose': 'Data volume ablation — 25%'},
    {'name': 'T2', 'data_pct': 0.50, 'rank': 16,
     'target_modules': ['q_proj','k_proj','v_proj','o_proj'],
     'lr': 2e-4, 'batch': 2, 'grad_accum': 4, 'epochs': 1,
     'purpose': 'Data volume ablation — 50%'},
    {'name': 'T3', 'data_pct': 1.00, 'rank': 16,
     'target_modules': ['q_proj','k_proj','v_proj','o_proj'],
     'lr': 2e-4, 'batch': 2, 'grad_accum': 4, 'epochs': 1,
     'purpose': 'Data volume ablation — 100%'},
    {'name': 'T4', 'data_pct': 1.00, 'rank': 32,
     'target_modules': ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
     'lr': 1e-4, 'batch': 2, 'grad_accum': 8, 'epochs': 2,
     'purpose': 'LoRA capacity — rank 32, extended target modules'},
    {'name': 'T5', 'data_pct': 1.00, 'rank': 64,
     'target_modules': ['q_proj','k_proj','v_proj','o_proj'],
     'lr': 5e-5, 'batch': 2, 'grad_accum': 8, 'epochs': 2,
     'purpose': 'LoRA capacity — rank 64'},
]


def run_sft_trials(ds_formatted, PROMPTS, gold_answers,
                   output_dir, stage_name):
    """Run all 5 ablation trials for one dataset. Returns (all_records, trial_summary)."""
    all_records, trial_summary = [], []

    for cfg in TRIAL_CONFIGS:
        print(f"\n{'='*60}")
        print(f"[{stage_name}] {cfg['name']}: {cfg['purpose']}")
        print('='*60)

        n = int(len(ds_formatted) * cfg['data_pct'])
        ds_train = ds_formatted.select(range(n))
        split    = ds_train.train_test_split(test_size=0.1, seed=42)
        print(f"  Training examples: {len(split['train'])} | Validation examples: {len(split['test'])}")

        model, tokenizer = FastLanguageModel.from_pretrained(
            BASE_MODEL, max_seq_length=MAX_SEQ_LEN,
            load_in_4bit=True, token=HF_TOKEN)
        model = FastLanguageModel.get_peft_model(
            model, r=cfg['rank'],
            target_modules=cfg['target_modules'],
            lora_alpha=cfg['rank'],
            lora_dropout=0.05, bias='none',
            use_gradient_checkpointing='unsloth')   # Unsloth's optimised GC; saves ~30% VRAM vs True

        adapter_path = str(output_dir / cfg['name'])

        trainer = SFTTrainer(
            model=model, tokenizer=tokenizer,
            train_dataset=split['train'], eval_dataset=split['test'],
            args=SFTConfig(
                output_dir=adapter_path,
                dataset_text_field='text',
                max_seq_length=MAX_SEQ_LEN,
                packing=False,
                per_device_train_batch_size=cfg['batch'],
                gradient_accumulation_steps=cfg['grad_accum'],
                learning_rate=cfg['lr'],
                num_train_epochs=cfg['epochs'],
                eval_strategy='epoch', save_strategy='epoch',
                fp16=use_fp16, bf16=use_bf16,
                logging_steps=10, disable_tqdm=True, report_to='none'),
        )
        try:
            from transformers.utils.notebook import NotebookProgressCallback
            trainer.remove_callback(NotebookProgressCallback)
        except Exception:
            pass

        train_result = trainer.train()
        val_loss     = trainer.evaluate()['eval_loss']

        model.save_pretrained(adapter_path)
        tokenizer.save_pretrained(adapter_path)
        print(f"  Train loss: {train_result.training_loss:.4f} | Validation loss: {val_loss:.4f}")

        print(f"  Evaluating on {len(PROMPTS)} benchmark prompts ...")
        records = eval_model_on_prompts(model, tokenizer, PROMPTS, gold_answers, cfg['name'])
        for r in records:
            r.update({'stage': stage_name, 'data_pct': cfg['data_pct'],
                      'rank': cfg['rank'], 'lr': cfg['lr'], 'val_loss': val_loss})
        all_records.extend(records)

        avg = sum(r['score'] for r in records) / len(records)
        trial_summary.append({
            'trial': cfg['name'], 'purpose': cfg['purpose'],
            'data_pct': f"{cfg['data_pct']*100:.0f}%",
            'rank': cfg['rank'], 'lr': cfg['lr'],
            'batch': cfg['batch'], 'epochs': cfg['epochs'],
            'val_loss': round(val_loss, 4),
            'avg_judge_score': round(avg, 2),
            'adapter_path': adapter_path,
        })
        print(f"  Mean judge score: {avg:.2f}/5.00")

        del model, trainer
        gc.collect()
        torch.cuda.empty_cache()

    return all_records, trial_summary


print('Environment and helper functions initialised.')

---
## Phase 1 — Baseline Evaluation

The 30-prompt benchmark is defined here along with gold-standard reference answers. Both `Qwen3-0.6B` and `Qwen3-1.7B` are evaluated without any fine-tuning to establish baseline performance. Prompts span four categories: general logic (G1–G5), mathematics (M1–M5), difficult reasoning (D1–D10), and coding (C1–C10). Each response is scored 1–5 by the Kimi-K2-Instruct LLM judge.

In [ ]:
PROMPTS = [
    # ── General Logic (5) ────────────────────────────────────────────────────
    {'id':'G1','type':'general','prompt':('Alice, Bob, and Carol each own exactly one pet: a cat, a dog, or a fish. '
     'Alice does not own the cat. Bob does not own the dog. Carol does not own the fish. Who owns which pet? Reason step by step.')},
    {'id':'G2','type':'general','prompt':('A bat and a ball together cost $1.10. The bat costs $1.00 more than the ball. '
     'How much does the ball cost? Show your reasoning carefully.')},
    {'id':'G3','type':'general','prompt':('In a room of 30 people, is it more likely than not that at least two share a birthday? '
     'Explain your reasoning step by step without necessarily computing the exact probability.')},
    {'id':'G4','type':'general','prompt':('If all Bloops are Razzies, and all Razzies are Lazzies, are all Bloops definitely Lazzies? '
     'What logical principle applies here? Explain step by step.')},
    {'id':'G5','type':'general','prompt':('A farmer has 17 sheep. All but 9 die. How many sheep are left? '
     'Explain your reasoning before giving the answer.')},
    # ── Math Word Problems (5) ────────────────────────────────────────────────
    {'id':'M1','type':'math','prompt':('A train travels at 60 km/h for 2 hours, then at 90 km/h for 1.5 hours. '
     'What is the total distance travelled? Show all working.')},
    {'id':'M2','type':'math','prompt':'Find the sum of all integers from 1 to 100. Show the formula you use and explain why it works.'},
    {'id':'M3','type':'math','prompt':('A store sells an item at a 20% discount. After the discount the price is $80. '
     'What was the original price? Show step-by-step working.')},
    {'id':'M4','type':'math','prompt':('Two pipes can fill a tank in 6 hours and 4 hours respectively. '
     'If both pipes are opened simultaneously, how long will it take to fill the tank? Show all reasoning.')},
    {'id':'M5','type':'math','prompt':'A sequence starts: 2, 6, 18, 54, ... What is the 8th term? Identify the pattern and show your working.'},
    # ── Difficult Reasoning (10) ──────────────────────────────────────────────
    {'id':'D1','type':'difficult','prompt':('On an island, knights always tell the truth and knaves always lie. '
     "You meet two inhabitants A and B. A says: 'At least one of us is a knave.' "
     'Is A a knight or a knave? What about B? Reason step by step.')},
    {'id':'D2','type':'difficult','prompt':('You are on a game show. There are 3 doors: behind one is a car, behind the other two are goats. '
     'You pick door 1. The host, who knows what is behind each door, opens door 3 to reveal a goat. '
     'He offers you the chance to switch to door 2. Should you switch? '
     'What is the probability of winning if you switch versus if you stay? Reason carefully.')},
    {'id':'D3','type':'difficult','prompt':('How many times do the hour hand and minute hand of a clock overlap '
     '(point in exactly the same direction) in a 24-hour period? '
     'Derive the answer mathematically rather than counting.')},
    {'id':'D4','type':'difficult','prompt':('A rubber ball is dropped from a height of 100 metres. '
     'Each time it bounces, it rises to two-thirds of the height from which it just fell. '
     'What is the total distance (up and down combined) the ball travels before coming to rest? '
     'Show all working and simplify your answer.')},
    {'id':'D5','type':'difficult','prompt':('A disease affects 1% of the population. A diagnostic test is 99% accurate: '
     'if you have the disease the test is positive 99% of the time; '
     'if you do not have the disease the test is negative 99% of the time. '
     'You test positive. What is the probability you actually have the disease? '
     "Show all working using Bayes' theorem.")},
    {'id':'D6','type':'difficult','prompt':('You flip a fair coin repeatedly until you get heads. '
     'You win $2^n where n is the number of flips required. '
     'What is the expected value of your winnings? '
     'Analyse the result and explain why it is surprising (this is the St. Petersburg Paradox).')},
    {'id':'D7','type':'difficult','prompt':('In how many ways can 8 people be seated around a circular table? '
     'Now suppose two specific people, Alice and Bob, must NOT sit next to each other. '
     'How many valid arrangements remain? Show all working.')},
    {'id':'D8','type':'difficult','prompt':('A snail is at the bottom of a 10-metre well. '
     'Each day it climbs 3 metres, but each night it slides back 2 metres. '
     'How many days does it take to reach the top? '
     'Now generalise: for a well of height h metres, daily climb c metres, nightly slide s metres '
     '(with c > s), derive a formula for the number of days.')},
    {'id':'D9','type':'difficult','prompt':('Alice and Bob take turns rolling a fair six-sided die. Alice goes first. '
     'The first person to roll a 6 wins. What is the probability that Alice wins? '
     'Express your answer as a simplified fraction and show all working.')},
    {'id':'D10','type':'difficult','prompt':('Prove that the square root of 2 is irrational using proof by contradiction. '
     'Then state whether the square root of any non-perfect-square positive integer is always '
     'irrational, and justify your answer.')},
    # ── Coding (10) ───────────────────────────────────────────────────────────
    {'id':'C1','type':'coding','prompt':('Write a Python function is_prime(n) that returns True if n is prime and False otherwise. '
     'It must handle all non-negative integers correctly. Explain your approach and state its time complexity.')},
    {'id':'C2','type':'coding','prompt':('Write a Python function fib(n) that returns the nth Fibonacci number '
     '(fib(0)=0, fib(1)=1). Implement it using memoization. '
     'Explain how memoization improves the time complexity compared to naive recursion, '
     'and state the complexity of both approaches.')},
    {'id':'C3','type':'coding','prompt':('Write a Python function two_sum(nums, target) that takes a list of integers and a target, '
     'and returns the indices [i, j] of the two numbers that add up to the target. '
     'Assume exactly one solution exists and you cannot use the same element twice. '
     'Achieve O(n) time complexity and explain how.')},
    {'id':'C4','type':'coding','prompt':('Write a Python function binary_search(arr, target) that searches for target in a sorted list. '
     'Return the index if found, or -1 if not found. '
     'Do not use any built-in search functions. '
     'Explain why binary search achieves O(log n) time complexity.')},
    {'id':'C5','type':'coding','prompt':('Write a Python function is_balanced(s) that takes a string containing only the characters '
     '( ) { } [ ] and returns True if all brackets are correctly nested and matched, False otherwise. '
     'Explain your approach and its time complexity.')},
    {'id':'C6','type':'coding','prompt':('Define a Python class ListNode for a singly linked list node (with val and next attributes). '
     'Then write a function reverse_linked_list(head) that reverses the list in-place and returns '
     'the new head. Walk through an example with the list 1 -> 2 -> 3 -> 4 to verify your solution.')},
    {'id':'C7','type':'coding','prompt':('Write a Python function permutations(lst) that returns a list of all permutations of the '
     'input list of unique integers. Do not use itertools. '
     'Explain the recursive approach and state its time complexity.')},
    {'id':'C8','type':'coding','prompt':('Write a Python function merge_sorted(a, b) that takes two sorted lists of integers and '
     'returns a new sorted list containing all elements from both. '
     'Do not use sort() or sorted(). Achieve O(m+n) time complexity and explain the approach.')},
    {'id':'C9','type':'coding','prompt':('Write a Python function shortest_path(grid, start, end) where grid is a 2D list of 0s '
     '(passable) and 1s (walls), and start/end are (row, col) tuples. '
     'Return the number of steps in the shortest path from start to end, or -1 if no path exists. '
     'Use BFS and explain why BFS guarantees the shortest path.')},
    {'id':'C10','type':'coding','prompt':('Write a Python function calculate(expression) that evaluates a string arithmetic expression '
     'containing non-negative integers and the operators +, -, *, // (integer division). '
     'The expression may contain spaces. Respect standard operator precedence (* and // before + and -). '
     "For example: calculate('3 + 5 * 2') should return 13. Explain your approach.")},
]

type_counts = {}
for p in PROMPTS:
    type_counts[p['type']] = type_counts.get(p['type'], 0) + 1
print(f'Defined {len(PROMPTS)} prompts: {type_counts}')

In [ ]:
gold_answers = {
    'G1': 'Two valid solutions exist:\n1. Alice -> dog, Bob -> fish, Carol -> cat\n2. Alice -> fish, Bob -> cat, Carol -> dog\nStep 1: Alice cannot own the cat, so she owns dog or fish.\nStep 2: If Alice owns dog: Bob owns fish or cat. Bob cannot own dog. Carol cannot own fish, so Carol owns cat, Bob owns fish. Valid.\nStep 3: If Alice owns fish: Bob and Carol have cat/dog. Bob cannot own dog, so Bob owns cat, Carol owns dog. Valid.',
    'G2': 'Let ball = x. Bat = x + 1.00. Together: 2x + 1.00 = 1.10, so 2x = 0.10, x = 0.05. The ball costs $0.05 and the bat costs $1.05.',
    'G3': 'Yes, more likely than not (~70.6% probability). The probability all 30 birthdays differ is (365/365)x(364/365)x...x(336/365) ≈ 0.294. So P(at least one shared) = 1 - 0.294 ≈ 0.706. With 30 people there are 435 pairs — many chances for a match.',
    'G4': 'Yes. This is the transitive property of class inclusion (syllogism). All Bloops are Razzies (Bloops ⊆ Razzies) and all Razzies are Lazzies (Razzies ⊆ Lazzies), therefore Bloops ⊆ Lazzies. Every Bloop is a Lazzie.',
    'G5': '9 sheep. "All but 9 die" means 9 survive. The phrase means every sheep except 9 has died.',
    'M1': 'Part 1: 60 km/h x 2 h = 120 km. Part 2: 90 km/h x 1.5 h = 135 km. Total = 120 + 135 = 255 km.',
    'M2': 'Sum = n(n+1)/2 = 100x101/2 = 5050. The formula works because pairing 1+100, 2+99, ..., 50+51 gives 50 pairs each summing to 101.',
    'M3': 'Let original price = x. After 20% discount: 0.8x = 80, so x = 100. The original price was $100.',
    'M4': 'Rates: 1/6 + 1/4 = 5/12 per hour. Time = 12/5 = 2.4 hours = 2 hours 24 minutes.',
    'M5': 'Geometric sequence, ratio = 3. a_n = 2 x 3^(n-1). a_8 = 2 x 3^7 = 2 x 2187 = 4374.',
    'D1': 'A is a knight, B is a knave. If A were a knave, his statement would be false (neither is a knave), contradicting A being a knave. So A must be a knight telling the truth — at least one is a knave — and since A is a knight, B is the knave.',
    'D2': 'Switch. Stay wins with probability 1/3; switch wins with probability 2/3. The host always reveals a goat, so the 2/3 probability originally on doors 2 and 3 collapses entirely onto door 2 after door 3 is opened.',
    'D3': '22 times. Relative speed = 5.5 deg/min. Time between overlaps = 360/5.5 = 720/11 min. Overlaps in 12h = 11. In 24h = 22.',
    'D4': '500 metres. First drop: 100m. Subsequent bounces: 200 x (2/3)/(1-2/3) = 400m. Total = 500m.',
    'D5': '50%. P(Disease|Positive) = (0.99x0.01)/(0.99x0.01 + 0.01x0.99) = 0.0099/0.0198 = 0.5. The rare disease (1%) makes false positives as common as true positives.',
    'D6': 'Expected value = sum of 2^k x (1/2)^k for k=1 to inf = sum of 1 = infinity. This is the St. Petersburg Paradox: infinite EV but most people would not pay more than ~$20-30 because large payouts have tiny probabilities and money has diminishing marginal utility.',
    'D7': 'Total circular arrangements: 7! = 5040. Alice+Bob adjacent: treat as unit, 6! x 2 = 1440. Alice+Bob NOT adjacent: 5040 - 1440 = 3600.',
    'D8': '8 days. Net gain per cycle = 1m. After 7 days at 7m. Day 8: climbs 3m to reach 10m. General formula: ceil((h-c)/(c-s)) + 1 days.',
    'D9': '6/11. P(Alice) = 1/6 + (25/36)P(Alice), so P(Alice)(11/36) = 1/6, P(Alice) = 6/11.',
    'D10': 'Proof: assume sqrt(2) = p/q in lowest terms. Then p^2 = 2q^2, so p is even (p=2k), giving 4k^2 = 2q^2, so q^2 = 2k^2, meaning q is also even — contradicting lowest terms. Therefore sqrt(2) is irrational. Yes, sqrt(n) is irrational for all non-perfect-square positive integers by the same argument.',
    'C1': 'def is_prime(n):\n    if n < 2: return False\n    if n == 2: return True\n    if n % 2 == 0: return False\n    i = 3\n    while i*i <= n:\n        if n % i == 0: return False\n        i += 2\n    return True\nTime complexity: O(sqrt(n)) — only checks odd divisors up to sqrt(n).',
    'C2': 'from functools import lru_cache\n@lru_cache(maxsize=None)\ndef fib(n):\n    if n <= 1: return n\n    return fib(n-1) + fib(n-2)\nNaive recursion: O(2^n). With memoization: O(n) time, O(n) space — each value computed once.',
    'C3': 'def two_sum(nums, target):\n    seen = {}\n    for i, num in enumerate(nums):\n        if target - num in seen:\n            return [seen[target-num], i]\n        seen[num] = i\nO(n): one pass, O(1) hash map lookup per element.',
    'C4': 'def binary_search(arr, target):\n    lo, hi = 0, len(arr)-1\n    while lo <= hi:\n        mid = (lo+hi)//2\n        if arr[mid] == target: return mid\n        elif arr[mid] < target: lo = mid+1\n        else: hi = mid-1\n    return -1\nO(log n): each step halves the search space.',
    'C5': 'def is_balanced(s):\n    stack = []\n    matching = {")": "(", "}": "{", "]": "["}\n    for ch in s:\n        if ch in "({[": stack.append(ch)\n        elif ch in ")}]":\n            if not stack or stack[-1] != matching[ch]: return False\n            stack.pop()\n    return len(stack) == 0\nO(n) time, O(n) space.',
    'C6': 'class ListNode:\n    def __init__(self, val=0, next=None):\n        self.val=val; self.next=next\ndef reverse_linked_list(head):\n    prev=None; curr=head\n    while curr:\n        nxt=curr.next; curr.next=prev; prev=curr; curr=nxt\n    return prev\nExample 1->2->3->4 becomes 4->3->2->1.',
    'C7': 'def permutations(lst):\n    if not lst: return [[]]\n    result = []\n    for i, elem in enumerate(lst):\n        for perm in permutations(lst[:i]+lst[i+1:]):\n            result.append([elem]+perm)\n    return result\nTime: O(n x n!) — n! permutations each of length n.',
    'C8': 'def merge_sorted(a, b):\n    result=[]; i=j=0\n    while i<len(a) and j<len(b):\n        if a[i]<=b[j]: result.append(a[i]); i+=1\n        else: result.append(b[j]); j+=1\n    result.extend(a[i:]); result.extend(b[j:])\n    return result\nO(m+n) — one pass through both lists.',
    'C9': 'from collections import deque\ndef shortest_path(grid, start, end):\n    if not grid or grid[start[0]][start[1]]==1: return -1\n    rows,cols=len(grid),len(grid[0])\n    q=deque([(start,0)]); visited={start}\n    for (r,c),steps in q:\n        if (r,c)==end: return steps\n        for dr,dc in[(-1,0),(1,0),(0,-1),(0,1)]:\n            nr,nc=r+dr,c+dc\n            if 0<=nr<rows and 0<=nc<cols and grid[nr][nc]==0 and (nr,nc) not in visited:\n                visited.add((nr,nc)); q.append(((nr,nc),steps+1))\n    return -1\nBFS guarantees shortest path: explores level by level.',
    'C10': 'import re\ndef calculate(expr):\n    tokens=re.findall(r"\\d+|[+\\-*\\/]{1,2}", expr.replace(" ",""))\n    stack=[int(tokens[0])]; i=1\n    while i<len(tokens):\n        op=tokens[i]; n=int(tokens[i+1])\n        if op=="*": stack[-1]*=n\n        elif op=="//": stack[-1]=int(stack[-1]/n)\n        elif op=="+": stack.append(n)\n        elif op=="-": stack.append(-n)\n        i+=2\n    return sum(stack)\nStack-based, single pass, O(n). Handles precedence: * and // applied immediately, + and - deferred.',
}

assert len(gold_answers) == 30
print(f'Gold answers loaded: {len(gold_answers)}')

In [ ]:
# ── Baseline evaluation — Qwen3-0.6B and Qwen3-1.7B ──────────────────────────
BASE_MODELS = ['Qwen/Qwen3-0.6B', 'Qwen/Qwen3-1.7B']

baseline_records = []

for model_id in BASE_MODELS:
    print(f'\nEvaluating baseline model: {model_id}')
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, token=HF_TOKEN,
        torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
        device_map='auto' if device == 'cuda' else 'cpu')
    model.eval()

    for p in PROMPTS:
        print(f'  {p["id"]}...', end=' ', flush=True)
        messages = [{'role': 'system', 'content': 'You are a helpful reasoning assistant. Think step by step.'},
                    {'role': 'user',   'content': p['prompt']}]
        ids = _apply_chat_template(tokenizer, messages).to(device)
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=512, temperature=0.1,
                                 do_sample=True, pad_token_id=tokenizer.eos_token_id)
        resp = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
        score, reason = judge_response(p['prompt'], gold_answers[p['id']], resp)
        baseline_records.append({
            'model': model_id, 'stage': 'baseline', 'trial': 'base',
            'prompt_id': p['id'], 'type': p['type'],
            'response': resp, 'score': score, 'reason': reason})
        print(f'score={score}/5')
        time.sleep(0.5)

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

df_baseline = pd.DataFrame(baseline_records)
df_baseline.to_csv(WORK_DIR / 'baseline_results.csv', index=False)

print('\n── Baseline Results by Model and Prompt Type ──')
print(df_baseline.groupby(['model','type'])['score'].mean().round(2).unstack())

# Save prompts and gold answers for downstream phases
with open(WORK_DIR / 'prompts.json', 'w') as f:
    json.dump(PROMPTS, f, indent=2)
with open(WORK_DIR / 'gold_answers.json', 'w') as f:
    json.dump(gold_answers, f, indent=2)

print('\nPhase 1 complete. Saved: baseline_results.csv, prompts.json, gold_answers.json')

---
## Phase 2 — LoRA Ablation: Dataset A (Crownelius)

**Dataset:** `Crownelius/Opus-4.6-Reasoning-3300x` — 2,160 training examples generated by Claude Opus 4.6, containing explicit chain-of-thought reasoning in `<think>` tags.  
**Format:** `problem` / `thinking` / `solution` columns; assistant turn formatted as `<think>{thinking}</think>\n{solution}`.

Five LoRA configurations (T1–T5) are trained sequentially, ablating data volume, LoRA rank, and target module coverage. The best-performing adapter (by judge score, then validation loss) is carried forward to Phase 4.

In [ ]:
ds_A_raw = load_dataset(DATASET_A_ID, split='train')
print(f'Dataset A: {len(ds_A_raw)} rows | columns: {list(ds_A_raw.features.keys())}')

tok_A = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)

def format_row_A(row):
    thinking = row.get('thinking', '').strip()
    solution = row.get('solution', '').strip()
    assistant = f'<think>{thinking}</think>\n{solution}' if thinking else solution
    messages = [
        {'role': 'system',    'content': 'You are a helpful reasoning assistant. Think step by step.'},
        {'role': 'user',      'content': row['problem'].strip()},
        {'role': 'assistant', 'content': assistant},
    ]
    return tok_A.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

ds_A = ds_A_raw.map(lambda r: {'text': format_row_A(r)})
print(f'Formatted {len(ds_A)} rows.')
print(ds_A[0]['text'][:400])

In [ ]:
# ── Reload PROMPTS / gold_answers from disk if kernel was restarted ───────────
if 'PROMPTS' not in dir() or 'gold_answers' not in dir():
    with open(WORK_DIR / 'prompts.json') as f:
        PROMPTS = json.load(f)
    with open(WORK_DIR / 'gold_answers.json') as f:
        gold_answers = json.load(f)
    print('Reloaded PROMPTS and gold_answers from disk.')

records_A, summary_A = run_sft_trials(
    ds_A, PROMPTS, gold_answers, ADAPTERS_A, 'sft_A')

df_summary_A = pd.DataFrame(summary_A)
best_A = df_summary_A.sort_values(
    ['avg_judge_score','val_loss'], ascending=[False,True]).iloc[0]

print('\n── Dataset A Ablation Results ──')
print(df_summary_A[['trial','data_pct','rank','lr','epochs','val_loss','avg_judge_score']].to_string(index=False))
print(f'\nBest trial: {best_A["trial"]} | mean score={best_A["avg_judge_score"]}/5.00 | val_loss={best_A["val_loss"]}')

df_summary_A.to_csv(WORK_DIR / 'trial_results_datasetA.csv', index=False)
pd.DataFrame(records_A).to_csv(WORK_DIR / 'all_scores_datasetA.csv', index=False)
with open(WORK_DIR / 'best_trial_datasetA.json', 'w') as f:
    json.dump(best_A.to_dict(), f, indent=2)

print('Phase 2 complete. Saved: trial_results_datasetA.csv, all_scores_datasetA.csv, best_trial_datasetA.json')

---
## Phase 3 — LoRA Ablation: Dataset B (Sky-T1)

**Dataset:** `NovaSky-AI/Sky-T1_data_17k` — 16,401 examples generated by QwQ-32B-Preview, containing structured long-form reasoning embedded with `<|begin_of_thought|>` / `<|end_of_thought|>` tags.  
**Format:** `conversations` list of `{from, value}` dicts; a 2,500-row stratified subset is used to maintain comparable scale to Dataset A.

The same five LoRA configurations are applied, enabling a controlled comparison of reasoning style (Claude Opus vs. QwQ) on an identical base model and evaluation benchmark.

In [ ]:
ds_B_raw = load_dataset(DATASET_B_ID, split='train')
print(f'Dataset B: {len(ds_B_raw)} rows | columns: {list(ds_B_raw.features.keys())}')

ds_B_sub = ds_B_raw.shuffle(seed=42).select(range(min(SUBSET_B_SIZE, len(ds_B_raw))))
print(f'Using {len(ds_B_sub)} rows.')

tok_B = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)

def format_row_B(row):
    convs = row['conversations']
    human = next((t['value'] for t in convs if t['from'] == 'human'), '')
    gpt   = next((t['value'] for t in convs if t['from'] == 'gpt'),   '')
    system = row.get('system', '').strip() or \
             'You are a helpful reasoning assistant. Think step by step before answering.'
    messages = [
        {'role': 'system',    'content': system},
        {'role': 'user',      'content': human.strip()},
        {'role': 'assistant', 'content': gpt.strip()},
    ]
    return tok_B.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

ds_B = ds_B_sub.map(lambda r: {'text': format_row_B(r)})
print(f'Formatted {len(ds_B)} rows.')
print(ds_B[0]['text'][:400])

In [ ]:
# ── Reload PROMPTS / gold_answers from disk if kernel was restarted ───────────
if 'PROMPTS' not in dir() or 'gold_answers' not in dir():
    with open(WORK_DIR / 'prompts.json') as f:
        PROMPTS = json.load(f)
    with open(WORK_DIR / 'gold_answers.json') as f:
        gold_answers = json.load(f)
    print('Reloaded PROMPTS and gold_answers from disk.')

records_B, summary_B = run_sft_trials(
    ds_B, PROMPTS, gold_answers, ADAPTERS_B, 'sft_B')

df_summary_B = pd.DataFrame(summary_B)
best_B = df_summary_B.sort_values(
    ['avg_judge_score','val_loss'], ascending=[False,True]).iloc[0]

print('\n── Dataset B Ablation Results ──')
print(df_summary_B[['trial','data_pct','rank','lr','epochs','val_loss','avg_judge_score']].to_string(index=False))
print(f'\nBest trial: {best_B["trial"]} | mean score={best_B["avg_judge_score"]}/5.00 | val_loss={best_B["val_loss"]}')

df_summary_B.to_csv(WORK_DIR / 'trial_results_datasetB.csv', index=False)
pd.DataFrame(records_B).to_csv(WORK_DIR / 'all_scores_datasetB.csv', index=False)
with open(WORK_DIR / 'best_trial_datasetB.json', 'w') as f:
    json.dump(best_B.to_dict(), f, indent=2)

print('Phase 3 complete. Saved: trial_results_datasetB.csv, all_scores_datasetB.csv, best_trial_datasetB.json')

---
## Phase 4 — Final Evaluation and Report

The best adapter from each dataset (selected in Phases 2 and 3) is loaded and evaluated against the base `Qwen3-0.6B` on all 30 benchmark prompts. Five summary tables are produced:

- **Table 1:** Dataset A trial results (all 5 configurations)
- **Table 2:** Dataset B trial results (all 5 configurations)
- **Table 3:** 3-way comparison — base vs. SFT-A vs. SFT-B across all 30 prompts
- **Table 4:** Average score broken down by prompt type (general, math, difficult, coding)
- **Table 5:** Cross-domain transfer — per-type score delta relative to baseline

In [ ]:
# ── Phase 4 recovery: reload all required state from disk ─────────────────────
# Safe to re-run even if variables already exist (no-ops if values are present).
if 'PROMPTS' not in dir() or 'gold_answers' not in dir():
    with open(WORK_DIR / 'prompts.json') as f:
        PROMPTS = json.load(f)
    with open(WORK_DIR / 'gold_answers.json') as f:
        gold_answers = json.load(f)
    print('Reloaded PROMPTS and gold_answers from disk.')

if 'best_A' not in dir():
    with open(WORK_DIR / 'best_trial_datasetA.json') as f:
        best_A = pd.Series(json.load(f))
    df_summary_A = pd.read_csv(WORK_DIR / 'trial_results_datasetA.csv')
    print(f'Reloaded best_A from disk: {best_A["trial"]} | score={best_A["avg_judge_score"]}')

if 'best_B' not in dir():
    with open(WORK_DIR / 'best_trial_datasetB.json') as f:
        best_B = pd.Series(json.load(f))
    df_summary_B = pd.read_csv(WORK_DIR / 'trial_results_datasetB.csv')
    print(f'Reloaded best_B from disk: {best_B["trial"]} | score={best_B["avg_judge_score"]}')

print(f'Phase 4 state ready — best_A={best_A["trial"]}, best_B={best_B["trial"]}')

In [ ]:
# ── Evaluate best SFT-A adapter on all 30 benchmark prompts ──────────────────
print(f'Loading best Dataset A adapter: {best_A["adapter_path"]}')
model_A, tok_A_inf = load_sft_model(best_A['adapter_path'])
eval_records_A = eval_model_on_prompts(model_A, tok_A_inf, PROMPTS, gold_answers, 'sft_A')
del model_A, tok_A_inf
gc.collect()
torch.cuda.empty_cache()
print('Dataset A evaluation complete.')

In [ ]:
# ── Evaluate best SFT-B adapter on all 30 benchmark prompts ──────────────────
print(f'Loading best Dataset B adapter: {best_B["adapter_path"]}')
model_B, tok_B_inf = load_sft_model(best_B['adapter_path'])
eval_records_B = eval_model_on_prompts(model_B, tok_B_inf, PROMPTS, gold_answers, 'sft_B')
del model_B, tok_B_inf
gc.collect()
torch.cuda.empty_cache()
print('Dataset B evaluation complete.')

In [ ]:
# ── Assemble complete evaluation dataframe ────────────────────────────────────
df_base_06 = pd.read_csv(WORK_DIR / 'baseline_results.csv')
df_base_06 = df_base_06[df_base_06['model'].str.contains('0.6B')].copy()
df_base_06['label'] = 'base'

df_eval = pd.concat([
    df_base_06[['label','prompt_id','type','score','reason','response']],
    pd.DataFrame(eval_records_A),
    pd.DataFrame(eval_records_B),
], ignore_index=True)

# ── Table 1 — Dataset A ablation results ─────────────────────────────────────
cols = ['trial','data_pct','rank','lr','epochs','val_loss','avg_judge_score']
print('\n══ TABLE 1 — Dataset A Ablation Results (Crownelius) ══')
print(df_summary_A[cols].to_string(index=False))
print(f'Best: {best_A["trial"]} | mean score={best_A["avg_judge_score"]}/5.00 | val_loss={best_A["val_loss"]}')

# ── Table 2 — Dataset B ablation results ─────────────────────────────────────
print('\n══ TABLE 2 — Dataset B Ablation Results (Sky-T1) ══')
print(df_summary_B[cols].to_string(index=False))
print(f'Best: {best_B["trial"]} | mean score={best_B["avg_judge_score"]}/5.00 | val_loss={best_B["val_loss"]}')

# ── Table 3 — 3-way comparison across all 30 prompts ─────────────────────────
pivot = df_eval.pivot_table(
    index='prompt_id', columns='label', values='score')[['base','sft_A','sft_B']]
pivot.loc['AVERAGE'] = pivot.mean().round(2)
print('\n══ TABLE 3 — 3-Way Comparison: Base vs. SFT-A vs. SFT-B (30 prompts) ══')
print(pivot.to_string())

# ── Table 4 — Per-type breakdown ──────────────────────────────────────────────
type_pivot = df_eval.groupby(['label','type'])['score'].mean().round(2).unstack()
type_pivot = type_pivot.reindex(['base','sft_A','sft_B'])
print('\n══ TABLE 4 — Mean Score by Prompt Category ══')
print(type_pivot.to_string())

# ── Table 5 — Cross-domain transfer ──────────────────────────────────────────
print('\n══ TABLE 5 — Cross-Domain Transfer (Δ relative to baseline) ══')
print(f'{"Category":<12} {"Base":>6} {"SFT-A":>7} {"SFT-B":>7}  {"SFT-A Δ":>10}  {"SFT-B Δ":>10}')
print('-' * 65)
cross_rows = []
for ptype in ['general','math','difficult','coding']:
    sub = df_eval[df_eval['type'] == ptype]
    b  = sub[sub['label']=='base' ]['score'].mean()
    a  = sub[sub['label']=='sft_A']['score'].mean()
    bb = sub[sub['label']=='sft_B']['score'].mean()
    print(f'{ptype:<12} {b:>6.2f} {a:>7.2f} {bb:>7.2f}  {a-b:>+10.2f}  {bb-b:>+10.2f}')
    cross_rows.append({'category': ptype, 'base': round(b,2),
                       'sft_A': round(a,2), 'sft_B': round(bb,2),
                       'sft_A_delta': round(a-b,2), 'sft_B_delta': round(bb-b,2)})

# ── Persist all outputs ───────────────────────────────────────────────────────
pivot.to_csv(WORK_DIR / 'comparison_3way.csv')
type_pivot.to_csv(WORK_DIR / 'comparison_by_type.csv')
pd.DataFrame(cross_rows).to_csv(WORK_DIR / 'cross_domain_eval.csv', index=False)
df_eval.to_csv(WORK_DIR / 'all_eval_records.csv', index=False)

print('\nPhase 4 complete. All output files saved to /kaggle/working/:')
print('  baseline_results.csv | trial_results_datasetA.csv | trial_results_datasetB.csv')
print('  comparison_3way.csv  | comparison_by_type.csv     | cross_domain_eval.csv')
print('  all_eval_records.csv | all_scores_datasetA.csv    | all_scores_datasetB.csv')